# DailyETLMain — Databricks Migration

Generated from SSIS package by the migration agent.

| Field | Value |
| --- | --- |
| Steps | 13 |
| Unresolved | 0 |
| Planner | deterministic |


## Load City Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_city_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetCityUpdates` |
| Target Objects | `Integration.City_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load City Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetCityUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedCityData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
# [set_table_name] (CONTROL)
table_name = "Integration.City_Staging"


# [get_last_cutoff] (CONTROL)
last_cutoff = get_last_etl_cutoff(
    table_name=table_name
)


# [get_lineage_key] (CONTROL)
lineage_key = get_lineage_key(
    table_name=table_name,
    cutoff_time=new_cutoff
)


# [clear_staging] (STAGING)
spark.sql("""
DELETE FROM Integration.City_Staging
""")


# [extract_to_staging] (EXTRACT)
# AI_REQUIRED
#
# Source procedure cannot be deterministically translated.
#
# Translation status:
# UNSUPPORTED_PROCEDURAL_SQL
#
# Unsupported/complex features:
# CURSOR, WHILE, TEMP_TABLE
#
# The original procedure is preserved in operation metadata.
#
# The migration agent must convert this procedure
# into executable Databricks logic.


# [resolve_dimension_keys] (LOOKUP_UPDATE)
dimension_targets = [
    "Dimension.City"
]

# Resolve dimension/surrogate keys according to
# the semantic analysis of the source procedure.

# Semantic operation:
# RESOLVE_DIMENSION_KEYS

# Required target dimensions:
# "Dimension.City"


# [insert_target] (INSERT)
# TODO: generate executable INSERT.
#
# Target:
# Dimension.City
#
# Semantic analysis:
# {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Dimension.City', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


# [complete_lineage] (UPDATE)
# Mark lineage record as successfully completed.
complete_lineage(lineage_key)


# [update_etl_cutoff] (UPDATE)
# Advance ETL cutoff after successful load.
update_etl_cutoff(
    table_name=table_name,
    cutoff_time=new_cutoff
)


## Load Customer Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_customer_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetCustomerUpdates` |
| Target Objects | `Integration.Customer_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- All source tables (Sales.Customers, Sales.Customers_Archive, Sales.BuyingGroups, Sales.BuyingGroups_Archive, Sales.CustomerCategories, Sales.CustomerCategories_Archive, Application.People) exist as Delta tables in Databricks and contain the columns ValidFrom (datetime) and ValidTo (datetime) implementing system‑versioned temporal history.
- Application.People also stores historical rows with ValidFrom/ValidTo. If it does not, the code will still run but will use only the current state for that dimension.
- The number of distinct change timestamps between the cut‑offs is small enough to be collected to the driver for iteration.


In [ ]:
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F
import pyspark.sql.window as W
from datetime import datetime

def get_customer_updates(spark: SparkSession, last_cutoff: str, new_cutoff: str) -> DataFrame:
    """Extract incremental customer dimension changes between two cut‑off timestamps.

    This implementation reproduces the behaviour of the source SQL Server procedure
    `Integration.GetCustomerUpdates` using Spark DataFrames on Databricks.

    Parameters
    ----------
    spark: SparkSession
        Active Spark session.
    last_cutoff: str
        Lower bound (exclusive) of the incremental window, e.g. "2023-01-01T00:00:00".
    new_cutoff: str
        Upper bound (inclusive) of the incremental window, e.g. "2023-01-31T23:59:59".

    Returns
    -------
    DataFrame
        Columns matching the staging table `Integration.Customer_Staging`:
        ["WWI Customer ID", "Customer", "Bill To Customer", "Category",
         "Buying Group", "Primary Contact", "Postal Code", "Valid From", "Valid To"]
    """
    # ---------------------------------------------------------------------
    # Assumptions (explicitly declared as required by the migration spec)
    # ---------------------------------------------------------------------
    # 1. All source tables are available as Delta tables with the same names
    #    used in the source procedure (e.g. "Sales.Customers", "Sales.Customers_Archive",
    #    "Sales.BuyingGroups", "Sales.BuyingGroups_Archive", "Sales.CustomerCategories",
    #    "Sales.CustomerCategories_Archive", "Application.People").
    # 2. Each of these tables implements system‑versioned temporal semantics via
    #    row‑level columns `ValidFrom` (datetime) and `ValidTo` (datetime).  The
    #    archive tables contain the same schema.
    # 3. The archive tables follow the pattern `{schema}.{table}_Archive` as
    #    described in the supplied `temporal_context`.
    # 4. `Application.People` also contains `ValidFrom`/`ValidTo` columns – this
    #    is required for point‑in‑time joins.  If it does not, the code will still
    #    run but will use the current state only (documented as a limitation).
    # 5. The constant `END_OF_TIME` is defined as the maximum datetime used in the
    #    source procedure.
    # ---------------------------------------------------------------------

    END_OF_TIME = "9999-12-31 23:59:59.9999999"
    INITIAL_LOAD_DATE = "2013-01-01"

    # ---------------------------------------------------------------------
    # Helper to create a unified view (current + archive) for a given table.
    # ---------------------------------------------------------------------
    def union_current_and_archive(current_tbl: str, archive_tbl: str) -> DataFrame:
        cur = spark.table(current_tbl)
        arch = spark.table(archive_tbl)
        return cur.unionByName(arch)

    # Unified tables with history
    customers = union_current_and_archive("Sales.Customers", "Sales.Customers_Archive")
    buying_groups = union_current_and_archive("Sales.BuyingGroups", "Sales.BuyingGroups_Archive")
    categories = union_current_and_archive("Sales.CustomerCategories", "Sales.CustomerCategories_Archive")
    people = spark.table("Application.People")  # assume it already contains history columns

    # ---------------------------------------------------------------------
    # 1. Gather change timestamps for each entity within the window.
    # ---------------------------------------------------------------------
    def change_timestamps(df: DataFrame, id_col: str) -> DataFrame:
        return (
            df.filter(
                (F.col("ValidFrom") > F.lit(last_cutoff))
                & (F.col("ValidFrom") <= F.lit(new_cutoff))
                & (F.col("ValidFrom") != F.lit(INITIAL_LOAD_DATE))
            )
            .select(F.col(id_col).alias("id"), F.col("ValidFrom").alias("valid_from"))
        )

    bg_changes = change_timestamps(buying_groups, "BuyingGroupID")
    cat_changes = change_timestamps(categories, "CustomerCategoryID")
    cust_changes = change_timestamps(customers, "CustomerID")

    # Union all timestamps (distinct) – order will be applied later per‑entity.
    all_timestamps = (
        bg_changes.select("valid_from")
        .union(cat_changes.select("valid_from"))
        .union(cust_changes.select("valid_from"))
        .distinct()
        .orderBy("valid_from")
    )

    # Collect timestamps to driver – expected to be a modest number (incremental).
    ts_list = [row["valid_from"] for row in all_timestamps.collect()]

    # ---------------------------------------------------------------------
    # 2. For each timestamp, materialise point‑in‑time snapshots and generate rows.
    # ---------------------------------------------------------------------
    result_rows = []

    for ts in ts_list:
        # Snapshot of each entity at the current timestamp
        cust_snapshot = customers.filter(
            (F.col("ValidFrom") <= F.lit(ts)) & (F.col("ValidTo") > F.lit(ts))
        )
        bg_snapshot = buying_groups.filter(
            (F.col("ValidFrom") <= F.lit(ts)) & (F.col("ValidTo") > F.lit(ts))
        )
        cat_snapshot = categories.filter(
            (F.col("ValidFrom") <= F.lit(ts)) & (F.col("ValidTo") > F.lit(ts))
        )
        people_snapshot = people.filter(
            (F.col("ValidFrom") <= F.lit(ts)) & (F.col("ValidTo") > F.lit(ts))
        )

        # -----------------------------------------------------------------
        # Buying‑Group changes – all customers belonging to the changed group.
        # -----------------------------------------------------------------
        changed_bg_ids = (
            bg_changes.filter(F.col("valid_from") == ts)
            .select(F.col("id").alias("BuyingGroupID"))
        )
        if changed_bg_ids.count() > 0:
            bg_join = (
                cust_snapshot.alias("c")
                .join(bg_snapshot.alias("bg"), F.col("c.BuyingGroupID") == F.col("bg.BuyingGroupID"))
                .join(cat_snapshot.alias("cc"), F.col("c.CustomerCategoryID") == F.col("cc.CustomerCategoryID"))
                .join(cust_snapshot.alias("bt"), F.col("c.BillToCustomerID") == F.col("bt.CustomerID"))
                .join(people_snapshot.alias("p"), F.col("c.PrimaryContactPersonID") == F.col("p.PersonID"))
                .join(changed_bg_ids, F.col("bg.BuyingGroupID") == F.col("BuyingGroupID"))
                .select(
                    F.col("c.CustomerID").alias("WWI Customer ID"),
                    F.col("c.CustomerName").alias("Customer"),
                    F.col("bt.CustomerName").alias("Bill To Customer"),
                    F.col("cc.CustomerCategoryName").alias("Category"),
                    F.col("bg.BuyingGroupName").alias("Buying Group"),
                    F.col("p.FullName").alias("Primary Contact"),
                    F.col("c.DeliveryPostalCode").alias("Postal Code"),
                    F.lit(ts).cast("timestamp").alias("Valid From"),
                    F.lit(None).cast("timestamp").alias("Valid To"),
                )
            )
            result_rows.append(bg_join)

        # ---------------------------------------------------------------
        # Customer‑Category changes – all customers belonging to the changed category.
        # ---------------------------------------------------------------
        changed_cat_ids = (
            cat_changes.filter(F.col("valid_from") == ts)
            .select(F.col("id").alias("CustomerCategoryID"))
        )
        if changed_cat_ids.count() > 0:
            cat_join = (
                cust_snapshot.alias("c")
                .join(bg_snapshot.alias("bg"), F.col("c.BuyingGroupID") == F.col("bg.BuyingGroupID"))
                .join(cat_snapshot.alias("cc"), F.col("c.CustomerCategoryID") == F.col("cc.CustomerCategoryID"))
                .join(cust_snapshot.alias("bt"), F.col("c.BillToCustomerID") == F.col("bt.CustomerID"))
                .join(people_snapshot.alias("p"), F.col("c.PrimaryContactPersonID") == F.col("p.PersonID"))
                .join(changed_cat_ids, F.col("cc.CustomerCategoryID") == F.col("CustomerCategoryID"))
                .select(
                    F.col("c.CustomerID").alias("WWI Customer ID"),
                    F.col("c.CustomerName").alias("Customer"),
                    F.col("bt.CustomerName").alias("Bill To Customer"),
                    F.col("cc.CustomerCategoryName").alias("Category"),
                    F.col("bg.BuyingGroupName").alias("Buying Group"),
                    F.col("p.FullName").alias("Primary Contact"),
                    F.col("c.DeliveryPostalCode").alias("Postal Code"),
                    F.lit(ts).cast("timestamp").alias("Valid From"),
                    F.lit(None).cast("timestamp").alias("Valid To"),
                )
            )
            result_rows.append(cat_join)

        # ---------------------------------------------------------------
        # Direct Customer changes – the specific customer row that changed.
        # ---------------------------------------------------------------
        changed_cust_ids = (
            cust_changes.filter(F.col("valid_from") == ts)
            .select(F.col("id").alias("CustomerID"))
        )
        if changed_cust_ids.count() > 0:
            cust_join = (
                cust_snapshot.alias("c")
                .join(bg_snapshot.alias("bg"), F.col("c.BuyingGroupID") == F.col("bg.BuyingGroupID"))
                .join(cat_snapshot.alias("cc"), F.col("c.CustomerCategoryID") == F.col("cc.CustomerCategoryID"))
                .join(cust_snapshot.alias("bt"), F.col("c.BillToCustomerID") == F.col("bt.CustomerID"))
                .join(people_snapshot.alias("p"), F.col("c.PrimaryContactPersonID") == F.col("p.PersonID"))
                .join(changed_cust_ids, F.col("c.CustomerID") == F.col("CustomerID"))
                .select(
                    F.col("c.CustomerID").alias("WWI Customer ID"),
                    F.col("c.CustomerName").alias("Customer"),
                    F.col("bt.CustomerName").alias("Bill To Customer"),
                    F.col("cc.CustomerCategoryName").alias("Category"),
                    F.col("bg.BuyingGroupName").alias("Buying Group"),
                    F.col("p.FullName").alias("Primary Contact"),
                    F.col("c.DeliveryPostalCode").alias("Postal Code"),
                    F.lit(ts).cast("timestamp").alias("Valid From"),
                    F.lit(None).cast("timestamp").alias("Valid To"),
                )
            )
            result_rows.append(cust_join)

    # ---------------------------------------------------------------------
    # 3. Union all generated rows into a single DataFrame.
    # ---------------------------------------------------------------------
    if not result_rows:
        # No changes – return an empty DataFrame with the correct schema.
        empty_schema = [
            "WWI Customer ID",
            "Customer",
            "Bill To Customer",
            "Category",
            "Buying Group",
            "Primary Contact",
            "Postal Code",
            "Valid From",
            "Valid To",
        ]
        return spark.createDataFrame([], schema=" , ".join([f"{c} string" for c in empty_schema]))

    changes_df = result_rows[0]
    for df in result_rows[1:]:
        changes_df = changes_df.unionByName(df)

    # ---------------------------------------------------------------------
    # 4. Compute the [Valid To] column – next Valid From for the same customer.
    # ---------------------------------------------------------------------
    w = (
        W.Window.partitionBy("WWI Customer ID").orderBy("Valid From")
    )
    changes_df = (
        changes_df.withColumn("next_valid_from", F.lead("Valid From").over(w))
        .withColumn(
            "Valid To",
            F.when(F.col("next_valid_from").isNotNull(), F.col("next_valid_from"))
            .otherwise(F.lit(END_OF_TIME).cast("timestamp")),
        )
        .drop("next_valid_from")
    )

    # ---------------------------------------------------------------------
    # 5. Return the final DataFrame ordered by Valid From (mirrors source ORDER BY).
    # ---------------------------------------------------------------------
    return changes_df.orderBy("Valid From")

# -------------------------------------------------------------------------
# Example usage (to be removed or adapted in production pipelines):
# spark = SparkSession.builder.getOrCreate()
# df_updates = get_customer_updates(spark, "2023-01-01 00:00:00", "2023-01-31 23:59:59")
# df_updates.write.format("delta").mode("append").saveAsTable("Integration.Customer_Staging")
# -------------------------------------------------------------------------

## Load Employee Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_employee_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetEmployeeUpdates` |
| Target Objects | `Integration.Employee_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Employee Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetEmployeeUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedEmployeeData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
# [set_table_name] (CONTROL)
table_name = "Integration.Employee_Staging"


# [get_last_cutoff] (CONTROL)
last_cutoff = get_last_etl_cutoff(
    table_name=table_name
)


# [get_lineage_key] (CONTROL)
lineage_key = get_lineage_key(
    table_name=table_name,
    cutoff_time=new_cutoff
)


# [clear_staging] (STAGING)
spark.sql("""
DELETE FROM Integration.Employee_Staging
""")


# [extract_to_staging] (EXTRACT)
# AI_REQUIRED
#
# Source procedure cannot be deterministically translated.
#
# Translation status:
# UNSUPPORTED_PROCEDURAL_SQL
#
# Unsupported/complex features:
# CURSOR, WHILE, TEMP_TABLE
#
# The original procedure is preserved in operation metadata.
#
# The migration agent must convert this procedure
# into executable Databricks logic.


# [resolve_dimension_keys] (LOOKUP_UPDATE)
dimension_targets = [
    "Dimension.Employee"
]

# Resolve dimension/surrogate keys according to
# the semantic analysis of the source procedure.

# Semantic operation:
# RESOLVE_DIMENSION_KEYS

# Required target dimensions:
# "Dimension.Employee"


# [insert_target] (INSERT)
# TODO: generate executable INSERT.
#
# Target:
# Dimension.Employee
#
# Semantic analysis:
# {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Dimension.Employee', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


# [complete_lineage] (UPDATE)
# Mark lineage record as successfully completed.
complete_lineage(lineage_key)


# [update_etl_cutoff] (UPDATE)
# Advance ETL cutoff after successful load.
update_etl_cutoff(
    table_name=table_name,
    cutoff_time=new_cutoff
)


## Load Movement Fact

| Field | Value |
| --- | --- |
| Step ID | `load_movement_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetMovementUpdates` |
| Target Objects | `Integration.Movement_Staging` |


In [ ]:
# [set_table_name] (CONTROL)
table_name = "Integration.Movement_Staging"


# [get_last_cutoff] (CONTROL)
last_cutoff = get_last_etl_cutoff(
    table_name=table_name
)


# [get_lineage_key] (CONTROL)
lineage_key = get_lineage_key(
    table_name=table_name,
    cutoff_time=new_cutoff
)


# [clear_staging] (STAGING)
spark.sql("""
DELETE FROM Integration.Movement_Staging
""")


# [extract_to_staging] (EXTRACT)
source_df = spark.sql("""
SELECT CAST(sit.TransactionOccurredWhen AS date) AS `Date Key`,
           sit.StockItemTransactionID AS `WWI Stock Item Transaction ID`,
           sit.InvoiceID AS `WWI Invoice ID`,
           sit.PurchaseOrderID AS `WWI Purchase Order ID`,
           CAST(sit.Quantity AS int) AS Quantity,
           sit.StockItemID AS `WWI Stock Item ID`,
           sit.CustomerID AS `WWI Customer ID`,
           sit.SupplierID AS `WWI Supplier ID`,
           sit.TransactionTypeID AS `WWI Transaction Type ID`,
           sit.TransactionOccurredWhen AS `Transaction Occurred When`
    FROM Warehouse.StockItemTransactions AS sit
    WHERE sit.LastEditedWhen > :LastCutoff
    AND sit.LastEditedWhen <= :NewCutoff
    ORDER BY sit.StockItemTransactionID
""")


# [resolve_dimension_keys] (LOOKUP_UPDATE)
dimension_targets = [
    "Dimension.Stock Item", "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type"
]

# Resolve dimension/surrogate keys according to
# the semantic analysis of the source procedure.

# Semantic operation:
# RESOLVE_DIMENSION_KEYS

# Required target dimensions:
# "Dimension.Stock Item", "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type"


# [merge_target] (MERGE)
# TODO: generate executable Delta MERGE.
#
# Target:
# Fact.Movement
#
# Semantic analysis:
# {'operation': 'MERGE_TARGET', 'type': 'MERGE', 'description': 'Merge staged records into the target fact or dimension table.', 'target': 'Fact.Movement', 'features': ['WHEN_MATCHED', 'WHEN_NOT_MATCHED', 'UPDATE', 'INSERT']}
#
# The source procedure definition is preserved
# in operation metadata for deterministic generation
# or AI-assisted refinement.


# [complete_lineage] (UPDATE)
# Mark lineage record as successfully completed.
complete_lineage(lineage_key)


# [update_etl_cutoff] (UPDATE)
# Advance ETL cutoff after successful load.
update_etl_cutoff(
    table_name=table_name,
    cutoff_time=new_cutoff
)


## Load Order Fact

| Field | Value |
| --- | --- |
| Step ID | `load_order_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetOrderUpdates` |
| Target Objects | `Integration.Order_Staging` |


In [ ]:
# [set_table_name] (CONTROL)
table_name = "Integration.Order_Staging"


# [get_last_cutoff] (CONTROL)
last_cutoff = get_last_etl_cutoff(
    table_name=table_name
)


# [get_lineage_key] (CONTROL)
lineage_key = get_lineage_key(
    table_name=table_name,
    cutoff_time=new_cutoff
)


# [clear_staging] (STAGING)
spark.sql("""
DELETE FROM Integration.Order_Staging
""")


# [extract_to_staging] (EXTRACT)
source_df = spark.sql("""
SELECT CAST(o.OrderDate AS date) AS `Order Date Key`,
           CAST(ol.PickingCompletedWhen AS date) AS `Picked Date Key`,
           o.OrderID AS `WWI Order ID`,
           o.BackorderOrderID AS `WWI Backorder ID`,
           ol.`Description`,
           pt.PackageTypeName AS Package,
           ol.Quantity AS Quantity,
           ol.UnitPrice AS `Unit Price`,
           ol.TaxRate AS `Tax Rate`,
           ROUND(ol.Quantity * ol.UnitPrice, 2) AS `Total Excluding Tax`,
           ROUND(ol.Quantity * ol.UnitPrice * ol.TaxRate / 100.0, 2) AS `Tax Amount`,
           ROUND(ol.Quantity * ol.UnitPrice, 2) + ROUND(ol.Quantity * ol.UnitPrice * ol.TaxRate / 100.0, 2) AS `Total Including Tax`,
           c.DeliveryCityID AS `WWI City ID`,
           c.CustomerID AS `WWI Customer ID`,
           ol.StockItemID AS `WWI Stock Item ID`,
           o.SalespersonPersonID AS `WWI Salesperson ID`,
           o.PickedByPersonID AS `WWI Picker ID`,
           CASE WHEN ol.LastEditedWhen > o.LastEditedWhen THEN ol.LastEditedWhen ELSE o.LastEditedWhen END AS `Last Modified When`
    FROM Sales.Orders AS o
    INNER JOIN Sales.OrderLines AS ol
    ON o.OrderID = ol.OrderID
    INNER JOIN Warehouse.PackageTypes AS pt
    ON ol.PackageTypeID = pt.PackageTypeID
    INNER JOIN Sales.Customers AS c
    ON c.CustomerID = o.CustomerID
    WHERE CASE WHEN ol.LastEditedWhen > o.LastEditedWhen THEN ol.LastEditedWhen ELSE o.LastEditedWhen END > :LastCutoff
    AND CASE WHEN ol.LastEditedWhen > o.LastEditedWhen THEN ol.LastEditedWhen ELSE o.LastEditedWhen END <= :NewCutoff
    ORDER BY o.OrderID
""")


# [resolve_dimension_keys] (LOOKUP_UPDATE)
dimension_targets = [
    "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"
]

# Resolve dimension/surrogate keys according to
# the semantic analysis of the source procedure.

# Semantic operation:
# RESOLVE_DIMENSION_KEYS

# Required target dimensions:
# "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"


# [delete_existing_target] (DELETE)
# TODO: generate executable DELETE.
#
# Target:
# Fact.Order
#
# Semantic analysis:
# {'operation': 'DELETE_EXISTING_TARGET', 'type': 'DELETE', 'description': 'Remove existing target records affected by the incoming staging records.', 'target': 'Fact.Order', 'features': ['DELETE', 'STAGING_KEY_FILTER']}


# [insert_target] (INSERT)
# TODO: generate executable INSERT.
#
# Target:
# Fact.Order
#
# Semantic analysis:
# {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Order', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


# [complete_lineage] (UPDATE)
# Mark lineage record as successfully completed.
complete_lineage(lineage_key)


# [update_etl_cutoff] (UPDATE)
# Advance ETL cutoff after successful load.
update_etl_cutoff(
    table_name=table_name,
    cutoff_time=new_cutoff
)


## Load Payment Method Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_payment_method_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetPaymentMethodUpdates` |
| Target Objects | `Integration.PaymentMethod_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- All source and destination tables are Delta tables accessible via Spark SQL.
- Archive table follows the pattern Application.PaymentMethods_Archive and contains the same schema (PaymentMethodID, PaymentMethodName, ValidFrom, ValidTo).
- ValidFrom and ValidTo columns are timestamps and follow row‑level validity semantics (timestamp >= ValidFrom AND timestamp < ValidTo).
- Staging table Integration.PaymentMethod_Staging already exists with columns [WWI Payment Method ID], [Payment Method], [Valid From], [Valid To].
- Dimension table Dimension.Payment Method is a Delta table with the same columns plus a [Lineage Key] column.
- Integration.Lineage contains columns [Lineage Key] (int), [Table Name] (string), [Data Load Completed] (timestamp, nullable), [Was Successful] (int), and [Source System Cutoff Time] (timestamp).
- Integration.[ETL Cutoff] contains columns [Table Name] (string) and [Cutoff Time] (timestamp).
- Workflow supplies ISO‑8601 strings for @LastCutoff and @NewCutoff.
- DeltaTable.merge is used to emulate the UPDATE of open‑ended dimension rows.


In [ ]:
# Databricks notebook – Load Payment Method Dimension
# ---------------------------------------------------
# This script implements the logical migration step "load_payment_method_dimension".
# It reproduces the behaviour of the source stored procedure Integration.GetPaymentMethodUpdates
# using Spark DataFrames and Delta Lake operations, and then runs the destination logic
# Integration.MigrateStagedPaymentMethodData.
# ---------------------------------------------------

from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col, lit, when, min as sql_min, max as sql_max, lead, coalesce, current_timestamp
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

# ---------------------------------------------------------------------------
# Configuration / constants (assumptions are documented at the end of the file)
# ---------------------------------------------------------------------------
END_OF_TIME = "9999-12-31 23:59:59.9999999"

# ---------------------------------------------------------------------------
# 1. Extract updated payment method data to staging (equivalent to the source
#    procedure Integration.GetPaymentMethodUpdates)
# ---------------------------------------------------------------------------

def get_payment_method_updates(last_cutoff: str, new_cutoff: str):
    """Extracts payment‑method changes between two cut‑off timestamps and writes
    the result to the staging table Integration.PaymentMethod_Staging.

    Parameters
    ----------
    last_cutoff: str
        ISO‑8601 timestamp representing the previous ETL watermark.
    new_cutoff: str
        ISO‑8601 timestamp representing the current ETL watermark.
    """
    # Load current and archive tables (assumed to be Delta tables)
    current_df = spark.table("Application.PaymentMethods")
    archive_df = spark.table("Application.PaymentMethods_Archive")

    # Union current and archive – both contain the validity columns ValidFrom/ValidTo
    all_versions = current_df.select(
        col("PaymentMethodID"),
        col("PaymentMethodName"),
        col("ValidFrom"),
        col("ValidTo")
    ).unionByName(
        archive_df.select(
            col("PaymentMethodID"),
            col("PaymentMethodName"),
            col("ValidFrom"),
            col("ValidTo")
        )
    )

    # -------------------------------------------------------------------
    # Step A – Identify change points (equivalent to the cursor SELECT … UNION ALL)
    # -------------------------------------------------------------------
    change_points = all_versions.filter(
        (col("ValidFrom") > lit(last_cutoff)) & (col("ValidFrom") <= lit(new_cutoff))
    ).select(
        col("PaymentMethodID"), col("ValidFrom")
    ).orderBy(col("ValidFrom"))

    # -------------------------------------------------------------------
    # Step B – For each change point, fetch the row version that was valid at that timestamp.
    # This replaces the procedural loop that used FOR SYSTEM_TIME AS OF.
    # -------------------------------------------------------------------
    # Join change points to the versioned table using the validity window.
    joined = change_points.alias("cp").join(
        all_versions.alias("av"),
        (col("cp.PaymentMethodID") == col("av.PaymentMethodID")) &
        (col("cp.ValidFrom") >= col("av.ValidFrom")) &
        (col("cp.ValidFrom") < col("av.ValidTo")),
        how="inner"
    ).select(
        col("av.PaymentMethodID").alias("WWI Payment Method ID"),
        col("av.PaymentMethodName").alias("Payment Method"),
        col("av.ValidFrom").alias("Valid From"),
        col("av.ValidTo").alias("Valid To")
    )

    # -------------------------------------------------------------------
    # Step C – Re‑calculate the Valid To column so that it reflects the next
    # change point for the same payment‑method ID, or END_OF_TIME if none.
    # -------------------------------------------------------------------
    w = Window.partitionBy(col("WWI Payment Method ID")).orderBy(col("Valid From"))
    final_df = joined.withColumn(
        "Calculated Valid To",
        coalesce(lead(col("Valid From")).over(w), lit(END_OF_TIME))
    ).drop("Valid To").withColumnRenamed("Calculated Valid To", "Valid To")

    # -------------------------------------------------------------------
    # Step D – Write the result to the staging table (append for incremental loads).
    # -------------------------------------------------------------------
    final_df.write.format("delta").mode("append").saveAsTable("Integration.PaymentMethod_Staging")

    return final_df

# ---------------------------------------------------------------------------
# 2. Migrate staged data into the dimension table (equivalent to
#    Integration.MigrateStagedPaymentMethodData)
# ---------------------------------------------------------------------------

def migrate_staged_payment_method_data():
    """Closes off existing open‑ended dimension rows, inserts the newly staged rows,
    and updates lineage and ETL‑cutoff tables.
    """
    # -------------------------------------------------------------------
    # Retrieve the current lineage key for this load (the row with a NULL
    # Data Load Completed flag, ordered by descending key).
    # -------------------------------------------------------------------
    lineage_key_row = spark.sql(
        """
        SELECT TOP 1 [Lineage Key]
        FROM Integration.Lineage
        WHERE [Table Name] = 'Payment Method' AND [Data Load Completed] IS NULL
        ORDER BY [Lineage Key] DESC
        """
    ).collect()
    if not lineage_key_row:
        raise RuntimeError("No pending lineage record found for 'Payment Method'.")
    lineage_key = lineage_key_row[0][0]

    # -------------------------------------------------------------------
    # Compute the earliest Valid From per WWI Payment Method ID in the staging table.
    # -------------------------------------------------------------------
    rows_to_close = spark.table("Integration.PaymentMethod_Staging") \
        .groupBy(col("WWI Payment Method ID")) \
        .agg(sql_min(col("Valid From")).alias("Valid From"))

    # -------------------------------------------------------------------
    # Update existing dimension rows that are still open (Valid To = END_OF_TIME)
    # to close them at the new Valid From.
    # -------------------------------------------------------------------
    dim_table = DeltaTable.forName(spark, "Dimension.Payment Method")
    dim_table.alias("dim").merge(
        rows_to_close.alias("rtc"),
        "dim.[WWI Payment Method ID] = rtc.[WWI Payment Method ID] AND dim.[Valid To] = '{}'".format(END_OF_TIME)
    ).whenMatchedUpdate(set={"dim.[Valid To]": col("rtc.[Valid From]")}).execute()

    # -------------------------------------------------------------------
    # Insert the staged rows into the dimension, attaching the lineage key.
    # -------------------------------------------------------------------
    staged_df = spark.table("Integration.PaymentMethod_Staging") \
        .withColumn("Lineage Key", lit(lineage_key))
    staged_df.write.format("delta").mode("append").saveAsTable("Dimension.Payment Method")

    # -------------------------------------------------------------------
    # Mark the lineage record as completed and successful.
    # -------------------------------------------------------------------
    spark.sql(
        f"""
        UPDATE Integration.Lineage
        SET [Data Load Completed] = CURRENT_TIMESTAMP(),
            [Was Successful] = 1
        WHERE [Lineage Key] = {lineage_key}
        """
    )

    # -------------------------------------------------------------------
    # Advance the ETL cutoff for the Payment Method table.
    # -------------------------------------------------------------------
    spark.sql(
        f"""
        UPDATE Integration.[ETL Cutoff]
        SET [Cutoff Time] = (
            SELECT [Source System Cutoff Time]
            FROM Integration.Lineage
            WHERE [Lineage Key] = {lineage_key}
        )
        WHERE [Table Name] = 'Payment Method'
        """
    )

# ---------------------------------------------------------------------------
# 3. Orchestrate the full step – this would be called by the workflow engine.
# ---------------------------------------------------------------------------

def run_load_payment_method_dimension(last_cutoff: str, new_cutoff: str):
    """Executes the full load step: extract → stage → migrate.
    """
    # Extract changes to staging
    get_payment_method_updates(last_cutoff, new_cutoff)
    # Migrate staged data into the dimension and update lineage / cutoff
    migrate_staged_payment_method_data()

# ---------------------------------------------------------------------------
# Example usage (the workflow would supply the actual cut‑off values):
# ---------------------------------------------------------------------------
# run_load_payment_method_dimension('2023-01-01T00:00:00', '2023-01-31T23:59:59')

# ---------------------------------------------------------------------------
# Assumptions
# ---------------------------------------------------------------------------
# - All source and destination tables are Delta tables accessible via Spark SQL.
# - The archive table follows the pattern Application.PaymentMethods_Archive and contains
#   the same schema (PaymentMethodID, PaymentMethodName, ValidFrom, ValidTo).
# - ValidFrom and ValidTo columns are of type timestamp and follow the semantics described
#   in the temporal_context (row is valid when timestamp >= ValidFrom AND timestamp < ValidTo).
# - The staging table Integration.PaymentMethod_Staging already exists with the columns:
#   [WWI Payment Method ID], [Payment Method], [Valid From], [Valid To].
# - The dimension table Dimension.Payment Method is a Delta table with the same columns plus
#   a [Lineage Key] column.
# - Integration.Lineage contains columns [Lineage Key] (int), [Table Name] (string),
#   [Data Load Completed] (timestamp, nullable), [Was Successful] (int), and
#   [Source System Cutoff Time] (timestamp).
# - Integration.[ETL Cutoff] contains columns [Table Name] (string) and [Cutoff Time] (timestamp).
# - The workflow supplies ISO‑8601 strings for the cut‑off parameters.
# - DeltaTable.update/merge semantics are used to emulate the UPDATE statements.
# ---------------------------------------------------------------------------
# Validation requirements
# ---------------------------------------------------------------------------
# - Verify that the number of rows written to Integration.PaymentMethod_Staging matches the
#   row count produced by the original stored procedure for the same cut‑off values.
# - After migration, ensure that for each WWI Payment Method ID the validity periods in
#   Dimension.Payment Method are continuous (no gaps) and non‑overlapping.
# - Confirm that the [Lineage Key] recorded in the inserted dimension rows matches the
#   key retrieved from Integration.Lineage.
# - Check that Integration.Lineage.[Data Load Completed] is populated and Integration.Lineage.[Was Successful] = 1.
# - Validate that Integration.[ETL Cutoff].[Cutoff Time] equals the source system cutoff time
#   from the processed lineage record.
# ---------------------------------------------------------------------------


## Load Purchase Fact

| Field | Value |
| --- | --- |
| Step ID | `load_purchase_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetPurchaseUpdates` |
| Target Objects | `Integration.Purchase_Staging` |


In [ ]:
# [set_table_name] (CONTROL)
table_name = "Integration.Purchase_Staging"


# [get_last_cutoff] (CONTROL)
last_cutoff = get_last_etl_cutoff(
    table_name=table_name
)


# [get_lineage_key] (CONTROL)
lineage_key = get_lineage_key(
    table_name=table_name,
    cutoff_time=new_cutoff
)


# [clear_staging] (STAGING)
spark.sql("""
DELETE FROM Integration.Purchase_Staging
""")


# [extract_to_staging] (EXTRACT)
source_df = spark.sql("""
SELECT CAST(po.OrderDate AS date) AS `Date Key`,
           po.PurchaseOrderID AS `WWI Purchase Order ID`,
           pol.OrderedOuters AS `Ordered Outers`,
           pol.OrderedOuters * si.QuantityPerOuter AS `Ordered Quantity`,
           pol.ReceivedOuters AS `Received Outers`,
           pt.PackageTypeName AS Package,
           pol.IsOrderLineFinalized AS `Is Order Finalized`,
           po.SupplierID AS `WWI Supplier ID`,
           pol.StockItemID AS `WWI Stock Item ID`,
           CASE WHEN pol.LastEditedWhen > po.LastEditedWhen THEN pol.LastEditedWhen ELSE po.LastEditedWhen END AS `Last Modified When`
    FROM Purchasing.PurchaseOrders AS po
    INNER JOIN Purchasing.PurchaseOrderLines AS pol
    ON po.PurchaseOrderID = pol.PurchaseOrderID
    INNER JOIN Warehouse.StockItems AS si
    ON pol.StockItemID = si.StockItemID
    INNER JOIN Warehouse.PackageTypes AS pt
    ON pol.PackageTypeID = pt.PackageTypeID
    WHERE CASE WHEN pol.LastEditedWhen > po.LastEditedWhen THEN pol.LastEditedWhen ELSE po.LastEditedWhen END > :LastCutoff
    AND CASE WHEN pol.LastEditedWhen > po.LastEditedWhen THEN pol.LastEditedWhen ELSE po.LastEditedWhen END <= :NewCutoff
    ORDER BY po.PurchaseOrderID
""")


# [resolve_dimension_keys] (LOOKUP_UPDATE)
dimension_targets = [
    "Dimension.Supplier", "Dimension.Stock Item"
]

# Resolve dimension/surrogate keys according to
# the semantic analysis of the source procedure.

# Semantic operation:
# RESOLVE_DIMENSION_KEYS

# Required target dimensions:
# "Dimension.Supplier", "Dimension.Stock Item"


# [delete_existing_target] (DELETE)
# TODO: generate executable DELETE.
#
# Target:
# Fact.Purchase
#
# Semantic analysis:
# {'operation': 'DELETE_EXISTING_TARGET', 'type': 'DELETE', 'description': 'Remove existing target records affected by the incoming staging records.', 'target': 'Fact.Purchase', 'features': ['DELETE', 'STAGING_KEY_FILTER']}


# [insert_target] (INSERT)
# TODO: generate executable INSERT.
#
# Target:
# Fact.Purchase
#
# Semantic analysis:
# {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Purchase', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


# [complete_lineage] (UPDATE)
# Mark lineage record as successfully completed.
complete_lineage(lineage_key)


# [update_etl_cutoff] (UPDATE)
# Advance ETL cutoff after successful load.
update_etl_cutoff(
    table_name=table_name,
    cutoff_time=new_cutoff
)


## Load Sale Fact

| Field | Value |
| --- | --- |
| Step ID | `load_sale_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetSaleUpdates` |
| Target Objects | `Integration.Sale_Staging` |


In [ ]:
# [set_table_name] (CONTROL)
table_name = "Integration.Sale_Staging"


# [get_last_cutoff] (CONTROL)
last_cutoff = get_last_etl_cutoff(
    table_name=table_name
)


# [get_lineage_key] (CONTROL)
lineage_key = get_lineage_key(
    table_name=table_name,
    cutoff_time=new_cutoff
)


# [clear_staging] (STAGING)
spark.sql("""
DELETE FROM Integration.Sale_Staging
""")


# [extract_to_staging] (EXTRACT)
source_df = spark.sql("""
SELECT CAST(i.InvoiceDate AS date) AS `Invoice Date Key`,
           CAST(i.ConfirmedDeliveryTime AS date) AS `Delivery Date Key`,
           i.InvoiceID AS `WWI Invoice ID`,
           il.`Description`,
           pt.PackageTypeName AS Package,
           il.Quantity,
           il.UnitPrice AS `Unit Price`,
           il.TaxRate AS `Tax Rate`,
           il.ExtendedPrice - il.TaxAmount AS `Total Excluding Tax`,
           il.TaxAmount AS `Tax Amount`,
           il.LineProfit AS Profit,
           il.ExtendedPrice AS `Total Including Tax`,
           CASE WHEN si.IsChillerStock = 0 THEN il.Quantity ELSE 0 END AS `Total Dry Items`,
           CASE WHEN si.IsChillerStock <> 0 THEN il.Quantity ELSE 0 END AS `Total Chiller Items`,
           c.DeliveryCityID AS `WWI City ID`,
           i.CustomerID AS `WWI Customer ID`,
           i.BillToCustomerID AS `WWI Bill To Customer ID`,
           il.StockItemID AS `WWI Stock Item ID`,
           i.SalespersonPersonID AS `WWI Saleperson ID`,
           CASE WHEN il.LastEditedWhen > i.LastEditedWhen THEN il.LastEditedWhen ELSE i.LastEditedWhen END AS `Last Modified When`
    FROM Sales.Invoices AS i
    INNER JOIN Sales.InvoiceLines AS il
    ON i.InvoiceID = il.InvoiceID
    INNER JOIN Warehouse.StockItems AS si
    ON il.StockItemID = si.StockItemID
    INNER JOIN Warehouse.PackageTypes AS pt
    ON il.PackageTypeID = pt.PackageTypeID
    INNER JOIN Sales.Customers AS c
    ON i.CustomerID = c.CustomerID
    INNER JOIN Sales.Customers AS bt
    ON i.BillToCustomerID = bt.CustomerID
    WHERE CASE WHEN il.LastEditedWhen > i.LastEditedWhen THEN il.LastEditedWhen ELSE i.LastEditedWhen END > :LastCutoff
    AND CASE WHEN il.LastEditedWhen > i.LastEditedWhen THEN il.LastEditedWhen ELSE i.LastEditedWhen END <= :NewCutoff
    ORDER BY i.InvoiceID, il.InvoiceLineID
""")


# [resolve_dimension_keys] (LOOKUP_UPDATE)
dimension_targets = [
    "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"
]

# Resolve dimension/surrogate keys according to
# the semantic analysis of the source procedure.

# Semantic operation:
# RESOLVE_DIMENSION_KEYS

# Required target dimensions:
# "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"


# [delete_existing_target] (DELETE)
# TODO: generate executable DELETE.
#
# Target:
# Fact.Sale
#
# Semantic analysis:
# {'operation': 'DELETE_EXISTING_TARGET', 'type': 'DELETE', 'description': 'Remove existing target records affected by the incoming staging records.', 'target': 'Fact.Sale', 'features': ['DELETE', 'STAGING_KEY_FILTER']}


# [insert_target] (INSERT)
# TODO: generate executable INSERT.
#
# Target:
# Fact.Sale
#
# Semantic analysis:
# {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Sale', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


# [complete_lineage] (UPDATE)
# Mark lineage record as successfully completed.
complete_lineage(lineage_key)


# [update_etl_cutoff] (UPDATE)
# Advance ETL cutoff after successful load.
update_etl_cutoff(
    table_name=table_name,
    cutoff_time=new_cutoff
)


## Load Stock Holding Fact

| Field | Value |
| --- | --- |
| Step ID | `load_stock_holding_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetStockHoldingUpdates` |
| Target Objects | `Integration.StockHolding_Staging` |


In [ ]:
# [set_table_name] (CONTROL)
table_name = "Integration.StockHolding_Staging"


# [get_last_cutoff] (CONTROL)
last_cutoff = get_last_etl_cutoff(
    table_name=table_name
)


# [get_lineage_key] (CONTROL)
lineage_key = get_lineage_key(
    table_name=table_name,
    cutoff_time=new_cutoff
)


# [clear_staging] (STAGING)
spark.sql("""
DELETE FROM Integration.StockHolding_Staging
""")


# [extract_to_staging] (EXTRACT)
source_df = spark.sql("""
SELECT sih.QuantityOnHand AS `Quantity On Hand`,
           sih.BinLocation AS `Bin Location`,
           sih.LastStocktakeQuantity AS `Last Stocktake Quantity`,
           sih.LastCostPrice AS `Last Cost Price`,
           sih.ReorderLevel AS `Reorder Level`,
           sih.TargetStockLevel AS `Target Stock Level`,
           sih.StockItemID AS `WWI Stock Item ID`
    FROM Warehouse.StockItemHoldings AS sih
    ORDER BY sih.StockItemID
""")


# [truncate_target] (TRUNCATE)
spark.sql("""
TRUNCATE TABLE Fact.Stock Holding
""")


# [insert_target] (INSERT)
# TODO: generate executable INSERT.
#
# Target:
# Fact.Stock Holding
#
# Semantic analysis:
# {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Stock Holding', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


# [complete_lineage] (UPDATE)
# Mark lineage record as successfully completed.
complete_lineage(lineage_key)


# [update_etl_cutoff] (UPDATE)
# Advance ETL cutoff after successful load.
update_etl_cutoff(
    table_name=table_name,
    cutoff_time=new_cutoff
)


## Load Stock Item Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_stock_item_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetStockItemUpdates` |
| Target Objects | `Integration.StockItem_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- Source tables (Warehouse.StockItems, Warehouse.StockItems_Archive, Warehouse.PackageTypes, Warehouse.PackageTypes_Archive, Warehouse.Colors, Warehouse.Colors_Archive) exist as Delta tables in Databricks and contain the columns ValidFrom and ValidTo required for row‑level temporal reconstruction.
- The Integration.ETL_Cutoff table already contains a row for Table Name = 'Stock Item'. If it does not, the code defaults the last cutoff to '1900‑01‑01'.
- A pending lineage record (Data Load Completed IS NULL) exists in Integration.Lineage for the Stock Item dimension; otherwise the step aborts.
- The mapping of the source column LeadTimeDays to the target column Tax Rate (as in the original procedure) is intentional and is preserved verbatim.
- Delta Lake tables support MERGE and UPDATE operations via the DeltaTable API; the environment has the delta‑core library available.
- The new cutoff is defined as the current timestamp at execution time – this matches typical incremental load patterns when a specific @NewCutoff is not supplied externally.


In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

# ---------------------------------------------------------------------------
# Configuration / constants
# ---------------------------------------------------------------------------
END_OF_TIME_STR = "9999-12-31 23:59:59.9999999"
END_OF_TIME = F.lit(END_OF_TIME_STR).cast("timestamp")
TABLE_NAME = "Stock Item"

# ---------------------------------------------------------------------------
# 1. Get the last ETL cutoff for the Stock Item dimension
# ---------------------------------------------------------------------------
etl_cutoff_df = (
    spark.table("Integration.ETL_Cutoff")
    .filter(F.col("Table Name") == TABLE_NAME)
    .select(F.col("Cutoff Time"))
)
last_cutoff_row = etl_cutoff_df.first()
if last_cutoff_row is None or last_cutoff_row["Cutoff Time"] is None:
    # If no prior cutoff exists, start from a very early date
    last_cutoff = F.lit("1900-01-01 00:00:00").cast("timestamp")
else:
    last_cutoff = F.lit(last_cutoff_row["Cutoff Time"]).cast("timestamp")

# ---------------------------------------------------------------------------
# 2. Determine the new cutoff (use the current timestamp as a pragmatic default)
# ---------------------------------------------------------------------------
new_cutoff = spark.sql("SELECT current_timestamp() as ts").first()["ts"]

# ---------------------------------------------------------------------------
# 3. Identify StockItem IDs that changed between the two cut‑offs
# ---------------------------------------------------------------------------
def changes_from(table_name: str):
    return (
        spark.table(table_name)
        .filter((F.col("ValidFrom") > last_cutoff) & (F.col("ValidFrom") <= new_cutoff))
        .select(
            F.col("StockItemID").alias("WWI Stock Item ID"),
            F.col("ValidFrom").alias("Valid From")
        )
    )

changes_archive = changes_from("Warehouse.StockItems_Archive")
changes_current = changes_from("Warehouse.StockItems")
changes = (
    changes_archive.unionByName(changes_current)
    .orderBy("Valid From")
)

# ---------------------------------------------------------------------------
# 4. Build a historical view of the source tables using row‑level validity windows
#    (FOR SYSTEM_TIME AS OF @ts  ⇢  ValidFrom <= @ts < ValidTo)
# ---------------------------------------------------------------------------
stock_items_hist = (
    spark.table("Warehouse.StockItems")
    .unionByName(spark.table("Warehouse.StockItems_Archive"))
    .select(*[F.col(c) for c in spark.table("Warehouse.StockItems").columns])
)
package_types_hist = (
    spark.table("Warehouse.PackageTypes")
    .unionByName(spark.table("Warehouse.PackageTypes_Archive"))
    .select(*[F.col(c) for c in spark.table("Warehouse.PackageTypes").columns])
)
colors_hist = (
    spark.table("Warehouse.Colors")
    .unionByName(spark.table("Warehouse.Colors_Archive"))
    .select(*[F.col(c) for c in spark.table("Warehouse.Colors").columns])
)

# ---------------------------------------------------------------------------
# 5. Join the change list with the historical snapshots to produce the staging rows
# ---------------------------------------------------------------------------
changes_alias = changes.alias("c")
si = stock_items_hist.alias("si")
spt = package_types_hist.alias("spt")
bpt = package_types_hist.alias("bpt")
col = colors_hist.alias("col")

staging_df = (
    changes_alias
    .join(
        si,
        (si.StockItemID == changes_alias["WWI Stock Item ID"])
        & (si.ValidFrom <= changes_alias["Valid From"])
        & (si.ValidTo > changes_alias["Valid From"]),
        "inner",
    )
    .join(
        spt,
        (si.UnitPackageID == spt.PackageTypeID)
        & (spt.ValidFrom <= changes_alias["Valid From"])
        & (spt.ValidTo > changes_alias["Valid From"]),
        "inner",
    )
    .join(
        bpt,
        (si.OuterPackageID == bpt.PackageTypeID)
        & (bpt.ValidFrom <= changes_alias["Valid From"])
        & (bpt.ValidTo > changes_alias["Valid From"]),
        "inner",
    )
    .join(
        col,
        (si.ColorID == col.ColorID)
        & (col.ValidFrom <= changes_alias["Valid From"])
        & (col.ValidTo > changes_alias["Valid From"]),
        "left",
    )
    .select(
        si.StockItemID.alias("WWI Stock Item ID"),
        si.StockItemName.alias("Stock Item"),
        col.ColorName.alias("Color"),
        spt.PackageTypeName.alias("Selling Package"),
        bpt.PackageTypeName.alias("Buying Package"),
        si.Brand,
        si.Size,
        si.LeadTimeDays.alias("Lead Time Days"),
        si.QuantityPerOuter.alias("Quantity Per Outer"),
        si.IsChillerStock.alias("Is Chiller Stock"),
        si.Barcode,
        # NOTE: The original procedure maps LeadTimeDays to Tax Rate – we preserve that mapping
        si.LeadTimeDays.alias("Tax Rate"),
        si.UnitPrice,
        si.RecommendedRetailPrice,
        si.TypicalWeightPerUnit,
        si.Photo,
        si.ValidFrom,
        si.ValidTo,
    )
)

# ---------------------------------------------------------------------------
# 6. Write the result to the staging table (overwrite for idempotency)
# ---------------------------------------------------------------------------
staging_df.write.format("delta").mode("overwrite").saveAsTable("Integration.StockItem_Staging")

# ---------------------------------------------------------------------------
# 7. Migration from staging to the dimension (close‑off + insert)
# ---------------------------------------------------------------------------
# 7a. Retrieve the pending lineage key
lineage_key_row = (
    spark.table("Integration.Lineage")
    .filter((F.col("Table Name") == TABLE_NAME) & (F.col("Data Load Completed").isNull()))
    .orderBy(F.col("Lineage Key").desc())
    .limit(1)
    .select("Lineage Key")
    .first()
)
if lineage_key_row is None:
    raise RuntimeError("No pending lineage record found for table 'Stock Item'.")
lineage_key = lineage_key_row["Lineage Key"]

# 7b. Determine the earliest Valid From per WWI Stock Item ID in the staging data (RowsToCloseOff)
rows_to_close = (
    spark.table("Integration.StockItem_Staging")
    .groupBy("WWI Stock Item ID")
    .agg(F.min("Valid From").alias("Valid From"))
)

# 7c. Close off existing dimension rows (set Valid To = new Valid From where currently open)
#    Using Delta MERGE for an in‑place update

dim_delta = DeltaTable.forName(spark, "Dimension.Stock Item")

# Build a temporary view that contains the rows to be updated with the new Valid To value
rows_to_update = (
    dim_delta.toDF().alias("d")
    .join(rows_to_close.alias("r"), dim_delta.toDF()["WWI Stock Item ID"] == rows_to_close["WWI Stock Item ID"], "inner")
    .filter(dim_delta.toDF()["Valid To"] == END_OF_TIME_STR)
    .select(
        "d.*",
        rows_to_close["Valid From"].alias("new_valid_to")
    )
)

# Perform the update via MERGE (match on primary key and open‑ended rows)
(dim_delta.alias("t")
 .merge(
     rows_to_update.alias("s"),
     "t.[WWI Stock Item ID] = s.[WWI Stock Item ID] AND t.[Valid To] = '{}'".format(END_OF_TIME_STR)
 )
 .whenMatchedUpdate(set={"Valid To": "s.new_valid_to"})
 .execute())

# 7d. Insert the new rows from staging, attaching the lineage key
staging_with_lineage = (
    spark.table("Integration.StockItem_Staging")
    .withColumn("Lineage Key", F.lit(lineage_key))
)

(dim_delta.alias("t")
 .merge(
     staging_with_lineage.alias("s"),
     "false"  # force insert‑only path
 )
 .whenNotMatchedInsertAll()
 .execute())

# ---------------------------------------------------------------------------
# 8. Mark the lineage record as completed
# ---------------------------------------------------------------------------
lineage_delta = DeltaTable.forName(spark, "Integration.Lineage")
lineage_delta.update(
    condition = F.col("Lineage Key") == lineage_key,
    set = {
        "Data Load Completed": F.current_timestamp(),
        "Was Successful": F.lit(1)
    }
)

# ---------------------------------------------------------------------------
# 9. Advance the ETL cutoff to the source‑system cutoff stored in the lineage record
# ---------------------------------------------------------------------------
source_cutoff = (
    spark.table("Integration.Lineage")
    .filter(F.col("Lineage Key") == lineage_key)
    .select(F.col("Source System Cutoff Time").alias("src_cutoff"))
    .first()["src_cutoff"]
)

etl_delta = DeltaTable.forName(spark, "Integration.ETL_Cutoff")
etl_delta.update(
    condition = F.col("Table Name") == TABLE_NAME,
    set = {"Cutoff Time": F.lit(source_cutoff)}
)

# ---------------------------------------------------------------------------
# End of migration step
# ---------------------------------------------------------------------------
print("Load Stock Item Dimension step completed successfully.")

## Load Supplier Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_supplier_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetSupplierUpdates` |
| Target Objects | `Integration.Supplier_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- Application.People table contains ValidFrom and ValidTo columns and therefore does not require a separate archive table.
- The number of distinct timestamps returned by the category‑change and supplier‑change queries is small enough to be collected to the driver for a Python loop.
- All source tables (Purchasing.SupplierCategories, Purchasing.SupplierCategories_Archive, Purchasing.Suppliers, Purchasing.Suppliers_Archive, Application.People) are available as Delta tables in the Databricks catalog with the same column names as in the source procedure.


In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window

def get_supplier_updates(spark, last_cutoff: str, new_cutoff: str):
    """Extract supplier changes between two cutoff timestamps and load them into the staging table.

    This implementation reproduces the logic of the source SQL Server procedure
    `Integration.GetSupplierUpdates` using Spark DataFrames on Databricks.
    It relies on row‑level temporal columns (ValidFrom, ValidTo) and archive tables
    following the `{schema}.{table}_Archive` pattern as described in the migration context.

    Parameters
    ----------
    spark : SparkSession
        Active Spark session.
    last_cutoff : str
        Lower bound of the incremental window (exclusive). Expected format: 'yyyy-MM-dd HH:mm:ss.SSSSSSS'.
    new_cutoff : str
        Upper bound of the incremental window (inclusive). Same format as ``last_cutoff``.
    """
    # ---------------------------------------------------------------------
    # 1. Prepare constants and helper functions
    # ---------------------------------------------------------------------
    end_of_time = F.lit('9999-12-31 23:59:59.9999999').cast('timestamp')
    initial_load_date = F.lit('2013-01-01').cast('date')
    last_cutoff_ts = F.to_timestamp(F.lit(last_cutoff))
    new_cutoff_ts = F.to_timestamp(F.lit(new_cutoff))

    def as_of(df, ts_col):
        """Return rows that are valid for the supplied timestamp (row‑level temporal filter)."""
        return df.filter((F.col('ValidFrom') <= ts_col) & (F.col('ValidTo') > ts_col))

    # ---------------------------------------------------------------------
    # 2. Load source tables (current + archive) – they must contain ValidFrom/ValidTo
    # ---------------------------------------------------------------------
    # Supplier categories (current + archive)
    supplier_categories_cur = spark.table('Purchasing.SupplierCategories')
    supplier_categories_arc = spark.table('Purchasing.SupplierCategories_Archive')
    supplier_categories = supplier_categories_cur.unionByName(supplier_categories_arc)

    # Suppliers (current + archive)
    suppliers_cur = spark.table('Purchasing.Suppliers')
    suppliers_arc = spark.table('Purchasing.Suppliers_Archive')
    suppliers = suppliers_cur.unionByName(suppliers_arc)

    # People – the context does not explicitly list an archive for Application.People.
    # We assume the current table contains the required validity columns.
    people = spark.table('Application.People')

    # ---------------------------------------------------------------------
    # 3. Build the list of supplier‑category changes (cursor replacement)
    # ---------------------------------------------------------------------
    cat_change_list = (
        supplier_categories
        .filter(
            (F.col('ValidFrom') > last_cutoff_ts) &
            (F.col('ValidFrom') <= new_cutoff_ts) &
            (F.col('ValidFrom').cast('date') != initial_load_date)
        )
        .select('SupplierCategoryID', 'ValidFrom')
        .orderBy('ValidFrom')
    )

    # Collect timestamps – expected to be a modest number, safe for driver side loop.
    cat_changes = cat_change_list.select('SupplierCategoryID', 'ValidFrom').collect()

    # ---------------------------------------------------------------------
    # 4. Build the list of supplier changes (second cursor replacement)
    # ---------------------------------------------------------------------
    supplier_change_list = (
        suppliers
        .filter(
            (F.col('ValidFrom') > last_cutoff_ts) &
            (F.col('ValidFrom') <= new_cutoff_ts)
        )
        .select('SupplierID', 'ValidFrom')
        .orderBy('ValidFrom')
    )
    supplier_changes = supplier_change_list.select('SupplierID', 'ValidFrom').collect()

    # ---------------------------------------------------------------------
    # 5. For each timestamp, materialise the rows that would have been inserted
    #    into the #SupplierChanges temp table in the original procedure.
    # ---------------------------------------------------------------------
    rows = []

    for row in cat_changes:
        cat_id = row['SupplierCategoryID']
        ts = row['ValidFrom']
        # point‑in‑time views of the three tables
        sup = as_of(suppliers, ts).alias('s')
        cat = as_of(supplier_categories, ts).alias('sc')
        per = as_of(people, ts).alias('p')
        joined = (
            sup.join(cat, sup['SupplierCategoryID'] == cat['SupplierCategoryID'])
               .join(per, sup['PrimaryContactPersonID'] == per['PersonID'])
               .filter(cat['SupplierCategoryID'] == cat_id)
               .select(
                   sup['SupplierID'].alias('WWI Supplier ID'),
                   sup['SupplierName'].alias('Supplier'),
                   cat['SupplierCategoryName'].alias('Category'),
                   per['FullName'].alias('Primary Contact'),
                   sup['SupplierReference'].alias('Supplier Reference'),
                   sup['PaymentDays'].alias('Payment Days'),
                   sup['DeliveryPostalCode'].alias('Postal Code'),
                   sup['ValidFrom'].alias('Valid From'),
                   sup['ValidTo'].alias('Valid To')
               )
        )
        rows.append(joined)

    for row in supplier_changes:
        sup_id = row['SupplierID']
        ts = row['ValidFrom']
        sup = as_of(suppliers, ts).alias('s')
        cat = as_of(supplier_categories, ts).alias('sc')
        per = as_of(people, ts).alias('p')
        joined = (
            sup.join(cat, sup['SupplierCategoryID'] == cat['SupplierCategoryID'])
               .join(per, sup['PrimaryContactPersonID'] == per['PersonID'])
               .filter(sup['SupplierID'] == sup_id)
               .select(
                   sup['SupplierID'].alias('WWI Supplier ID'),
                   sup['SupplierName'].alias('Supplier'),
                   cat['SupplierCategoryName'].alias('Category'),
                   per['FullName'].alias('Primary Contact'),
                   sup['SupplierReference'].alias('Supplier Reference'),
                   sup['PaymentDays'].alias('Payment Days'),
                   sup['DeliveryPostalCode'].alias('Postal Code'),
                   sup['ValidFrom'].alias('Valid From'),
                   sup['ValidTo'].alias('Valid To')
               )
        )
        rows.append(joined)

    # Union all rows collected from the two loops
    if rows:
        supplier_changes_df = rows[0]
        for df in rows[1:]:
            supplier_changes_df = supplier_changes_df.unionByName(df)
    else:
        # No changes – create an empty DataFrame with the expected schema
        supplier_changes_df = spark.createDataFrame([], schema='WWI Supplier ID INT, Supplier STRING, Category STRING, `Primary Contact` STRING, `Supplier Reference` STRING, `Payment Days` INT, `Postal Code` STRING, `Valid From` TIMESTAMP, `Valid To` TIMESTAMP')

    # ---------------------------------------------------------------------
    # 6. Compute the [Valid To] column as the next [Valid From] for the same supplier
    # ---------------------------------------------------------------------
    w = Window.partitionBy('WWI Supplier ID').orderBy('Valid From')
    supplier_changes_df = (
        supplier_changes_df
        .withColumn('next_valid_from', F.lead('Valid From').over(w))
        .withColumn('Valid To', F.coalesce(F.col('next_valid_from'), end_of_time))
        .drop('next_valid_from')
    )

    # ---------------------------------------------------------------------
    # 7. Load into the staging table (append semantics – the downstream procedure
    #    will handle deduplication / merging).
    # ---------------------------------------------------------------------
    supplier_changes_df.write.format('delta').mode('append').saveAsTable('Integration.Supplier_Staging')

    return supplier_changes_df

# -------------------------------------------------------------------------
# Example usage (to be removed or adapted in production pipelines):
# -------------------------------------------------------------------------
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()
# df = get_supplier_updates(spark, '2023-01-01 00:00:00.0000000', '2023-01-31 23:59:59.9999999')
# df.show()


## Load Transaction Fact

| Field | Value |
| --- | --- |
| Step ID | `load_transaction_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetTransactionUpdates` |
| Target Objects | `Integration.Transaction_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Transaction Fact
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['INCREMENTAL_COMPOSITE']
#
# Complexity signals:
# ['UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetTransactionUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedTransactionData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [ ]:
# [set_table_name] (CONTROL)
table_name = "Integration.Transaction_Staging"


# [get_last_cutoff] (CONTROL)
last_cutoff = get_last_etl_cutoff(
    table_name=table_name
)


# [get_lineage_key] (CONTROL)
lineage_key = get_lineage_key(
    table_name=table_name,
    cutoff_time=new_cutoff
)


# [clear_staging] (STAGING)
spark.sql("""
DELETE FROM Integration.Transaction_Staging
""")


# [extract_to_staging] (EXTRACT)
source_df = spark.sql("""
SELECT CAST(ct.TransactionDate AS date) AS `Date Key`,
           ct.CustomerTransactionID AS `WWI Customer Transaction ID`,
           CAST(NULL AS int) AS `WWI Supplier Transaction ID`,
           ct.InvoiceID AS `WWI Invoice ID`,
           CAST(NULL AS int) AS `WWI Purchase Order ID`,
           CAST(NULL AS nvarchar(20)) AS `Supplier Invoice Number`,
           ct.AmountExcludingTax AS `Total Excluding Tax`,
           ct.TaxAmount AS `Tax Amount`,
           ct.TransactionAmount AS `Total Including Tax`,
           ct.OutstandingBalance AS `Outstanding Balance`,
           ct.IsFinalized AS `Is Finalized`,
           COALESCE(i.CustomerID, ct.CustomerID) AS `WWI Customer ID`,
           ct.CustomerID AS `WWI Bill To Customer ID`,
           CAST(NULL AS int) AS `WWI Supplier ID`,
           ct.TransactionTypeID AS `WWI Transaction Type ID`,
           ct.PaymentMethodID AS `WWI Payment Method ID`,
           ct.LastEditedWhen AS `Last Modified When`
    FROM Sales.CustomerTransactions AS ct
    LEFT OUTER JOIN Sales.Invoices AS i
    ON ct.InvoiceID = i.InvoiceID
    WHERE ct.LastEditedWhen > :LastCutoff
    AND ct.LastEditedWhen <= :NewCutoff

    UNION ALL

    SELECT CAST(st.TransactionDate AS date) AS `Date Key`,
           CAST(NULL AS int) AS `WWI Customer Transaction ID`,
           st.SupplierTransactionID AS `WWI Supplier Transaction ID`,
           CAST(NULL AS int) AS `WWI Invoice ID`,
           st.PurchaseOrderID AS `WWI Purchase Order ID`,
           st.SupplierInvoiceNumber AS `Supplier Invoice Number`,
           st.AmountExcludingTax AS `Total Excluding Tax`,
           st.TaxAmount AS `Tax Amount`,
           st.TransactionAmount AS `Total Including Tax`,
           st.OutstandingBalance AS `Outstanding Balance`,
           st.IsFinalized AS `Is Finalized`,
           CAST(NULL AS int) AS `WWI Customer ID`,
           CAST(NULL AS int) AS `WWI Bill To Customer ID`,
           st.SupplierID AS `WWI Supplier ID`,
           st.TransactionTypeID AS `WWI Transaction Type ID`,
           st.PaymentMethodID AS `WWI Payment Method ID`,
           st.LastEditedWhen AS `Last Modified When`
    FROM Purchasing.SupplierTransactions AS st
    WHERE st.LastEditedWhen > :LastCutoff
    AND st.LastEditedWhen <= :NewCutoff
""")


# [resolve_dimension_keys] (LOOKUP_UPDATE)
dimension_targets = [
    "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type", "Dimension.Payment Method"
]

# Resolve dimension/surrogate keys according to
# the semantic analysis of the source procedure.

# Semantic operation:
# RESOLVE_DIMENSION_KEYS

# Required target dimensions:
# "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type", "Dimension.Payment Method"


# [insert_target] (INSERT)
# TODO: generate executable INSERT.
#
# Target:
# Fact.Transaction
#
# Semantic analysis:
# {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Transaction', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


# [complete_lineage] (UPDATE)
# Mark lineage record as successfully completed.
complete_lineage(lineage_key)


# [update_etl_cutoff] (UPDATE)
# Advance ETL cutoff after successful load.
update_etl_cutoff(
    table_name=table_name,
    cutoff_time=new_cutoff
)


## Load Transaction Type Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_transaction_type_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetTransactionTypeUpdates` |
| Target Objects | `Integration.TransactionType_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- Source tables Application.TransactionTypes and Application.TransactionTypes_Archive contain columns TransactionTypeID, TransactionTypeName, ValidFrom, ValidTo.
- Integration.ETL Cutoff table has columns Table Name (string) and Cutoff Time (timestamp).
- Integration.Lineage table has columns Lineage Key (int), Table Name (string), Data Load Completed (timestamp, nullable), Was Successful (int), Source System Cutoff Time (timestamp).
- Integration.TransactionType_Staging table exists with the same schema as the result of the extraction step.
- Dimension.Transaction Type table exists with columns matching the staging schema plus a Lineage Key column.
- The new cutoff can be derived from the current timestamp at runtime.
- Spark SQL UPDATE ... FROM syntax is supported in the Databricks runtime version used.


In [ ]:
import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def get_transaction_type_updates(spark: SparkSession, last_cutoff: datetime.datetime, new_cutoff: datetime.datetime):
    """Replicates the logic of Integration.GetTransactionTypeUpdates.

    Parameters
    ----------
    spark: SparkSession
        Active Spark session.
    last_cutoff: datetime
        The previous ETL watermark.
    new_cutoff: datetime
        The current ETL watermark (typically the time the job runs).

    Returns
    -------
    DataFrame
        Columns: `WWI Transaction Type ID`, `Transaction Type`, `Valid From`, `Valid To`
    """
    # Load current and archive tables.  The context guarantees they contain
    # TransactionTypeID, TransactionTypeName, ValidFrom, ValidTo.
    df_current = spark.table("Application.TransactionTypes")
    df_archive = spark.table("Application.TransactionTypes_Archive")

    # Union current and archive to emulate system‑time view.
    df_all = df_current.unionByName(df_archive)

    # Identify rows whose ValidFrom falls within the incremental window.
    change_df = (
        df_all
        .select("TransactionTypeID", "ValidFrom")
        .where((F.col("ValidFrom") > F.lit(last_cutoff)) & (F.col("ValidFrom") <= F.lit(new_cutoff)))
        .distinct()
        .orderBy("ValidFrom")
    )

    # For each change timestamp retrieve the row version that was active at that time.
    # This is equivalent to the FOR SYSTEM_TIME AS OF @ValidFrom query.
    # Condition: ValidFrom <= @ValidFrom < ValidTo
    changes = change_df.alias("c")
    all_rows = df_all.alias("a")
    result = (
        changes.join(
            all_rows,
            (F.col("c.TransactionTypeID") == F.col("a.TransactionTypeID")) &
            (F.col("c.ValidFrom") >= F.col("a.ValidFrom")) &
            (F.col("c.ValidFrom") < F.col("a.ValidTo")),
            "inner"
        )
        .select(
            F.col("a.TransactionTypeID").alias("WWI Transaction Type ID"),
            F.col("a.TransactionTypeName").alias("Transaction Type"),
            F.col("c.ValidFrom").alias("Valid From"),
            F.col("a.ValidTo").alias("Valid To")
        )
    )

    # Compute the correct Valid To for each row by looking at the next Valid From for the same ID.
    w = Window.partitionBy("WWI Transaction Type ID").orderBy("Valid From")
    result_with_next = result.withColumn("next_valid_from", F.lead("Valid From").over(w))
    end_of_time_literal = F.lit("9999-12-31 23:59:59.999999")
    final_df = (
        result_with_next
        .withColumn(
            "Valid To",
            F.when(F.col("next_valid_from").isNotNull(), F.col("next_valid_from")).otherwise(end_of_time_literal)
        )
        .drop("next_valid_from")
        .orderBy("Valid From")
    )
    return final_df


def load_transaction_type_dimension(spark: SparkSession):
    """Orchestrates the full load of the Transaction Type dimension.

    Steps (mirroring the original workflow):
    1. GET_LAST_CUTOFF – read the current cutoff for the Transaction Type table.
    2. SET_TABLE_NAME – (implicit in the queries, the table name is 'Transaction Type').
    3. EXTRACT_TO_STAGING – run the extraction logic and write to the staging table.
    4. GET_LINEAGE_KEY – obtain the pending lineage key.
    5. MIGRATE_STAGED_DATA – close off existing rows, insert new rows, update lineage and cutoff.
    6. CLEAR_STAGING – truncate the staging table.
    """
    # ---------------------------------------------------------------------
    # 1. GET_LAST_CUTOFF
    # ---------------------------------------------------------------------
    cutoff_row = (
        spark.table("Integration.ETL Cutoff")
        .filter(F.col("Table Name") == "Transaction Type")
        .select("Cutoff Time")
        .orderBy(F.col("Cutoff Time").desc())
        .limit(1)
        .collect()
    )
    if not cutoff_row:
        raise ValueError("No cutoff record found for 'Transaction Type' in Integration.ETL Cutoff.")
    last_cutoff = cutoff_row[0]["Cutoff Time"]

    # New cutoff – use the current timestamp as the watermark for this run.
    new_cutoff = spark.sql("SELECT current_timestamp() AS now").collect()[0]["now"]

    # ---------------------------------------------------------------------
    # 2. EXTRACT_TO_STAGING
    # ---------------------------------------------------------------------
    updates_df = get_transaction_type_updates(spark, last_cutoff, new_cutoff)
    # Write (overwrite) to the staging table.
    updates_df.write.mode("overwrite").saveAsTable("Integration.TransactionType_Staging")

    # ---------------------------------------------------------------------
    # 3. GET_LINEAGE_KEY
    # ---------------------------------------------------------------------
    lineage_row = (
        spark.table("Integration.Lineage")
        .filter((F.col("Table Name") == "Transaction Type") & (F.col("Data Load Completed").isNull()))
        .orderBy(F.col("Lineage Key").desc())
        .limit(1)
        .collect()
    )
    if not lineage_row:
        raise ValueError("No pending lineage record found for 'Transaction Type' in Integration.Lineage.")
    lineage_key = lineage_row[0]["Lineage Key"]

    # ---------------------------------------------------------------------
    # 4. MIGRATE_STAGED_DATA – translate the destination procedure logic.
    # ---------------------------------------------------------------------
    # a) Close off existing rows in the dimension where they are still open.
    spark.sql("""
        WITH RowsToCloseOff AS (
            SELECT pm.`WWI Transaction Type ID`, MIN(pm.`Valid From`) AS `Valid From`
            FROM Integration.TransactionType_Staging AS pm
            GROUP BY pm.`WWI Transaction Type ID`
        )
        UPDATE Dimension.`Transaction Type` AS d
        SET d.`Valid To` = r.`Valid From`
        FROM RowsToCloseOff AS r
        WHERE d.`WWI Transaction Type ID` = r.`WWI Transaction Type ID`
          AND d.`Valid To` = TIMESTAMP('9999-12-31 23:59:59.999999')
    """)

    # b) Insert new rows from staging into the dimension.
    spark.sql(f"""
        INSERT INTO Dimension.`Transaction Type` (
            `WWI Transaction Type ID`, `Transaction Type`, `Valid From`, `Valid To`, `Lineage Key`
        )
        SELECT `WWI Transaction Type ID`, `Transaction Type`, `Valid From`, `Valid To`, {lineage_key}
        FROM Integration.TransactionType_Staging
    """)

    # c) Mark lineage as completed.
    spark.sql(f"""
        UPDATE Integration.Lineage
        SET `Data Load Completed` = current_timestamp(),
            `Was Successful` = 1
        WHERE `Lineage Key` = {lineage_key}
    """)

    # d) Advance the ETL cutoff.
    spark.sql(f"""
        UPDATE Integration.`ETL Cutoff`
        SET `Cutoff Time` = (
            SELECT `Source System Cutoff Time`
            FROM Integration.Lineage
            WHERE `Lineage Key` = {lineage_key}
        )
        WHERE `Table Name` = 'Transaction Type'
    """)

    # ---------------------------------------------------------------------
    # 5. CLEAR_STAGING
    # ---------------------------------------------------------------------
    spark.sql("TRUNCATE TABLE Integration.TransactionType_Staging")

    # End of load – if this were a real transaction we would commit here.
    # Spark writes are atomic per statement; no explicit COMMIT needed.

# Example entry point (Databricks notebooks typically expose a Spark session as `spark`).
# load_transaction_type_dimension(spark)
